# Mixer-SNN vs DMP-SNN — benchmark trên Colab GPU

Chạy lần lượt từ trên xuống. Mỗi lần chạy xong một ô, **lưu notebook (Ctrl+S)** để Claude đọc được output.

* Kết quả (`results/*.json`) và checkpoint lưu trên Google Drive → Colab ngắt kết nối thì chạy lại đúng ô đó, run dang dở sẽ **tự resume**, run đã xong sẽ **tự bỏ qua**.
* Thứ tự ưu tiên: A (bảng chính, seed 0) → B (delayed recall) → C (ablation) → D (thêm seed 1, 2).
* Ô cuối cùng in toàn bộ kết quả dạng gọn để dán/đọc lại.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Lưu kết quả lên Google Drive (nếu mount lỗi trong VS Code thì kết quả nằm ở /content/bench_out — vẫn đọc được qua ô cuối)
import os
OUT = '/content/bench_out'
DATA = '/content/data'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/XLDL_bench'
except Exception as e:
    print('Drive mount failed:', e)
os.makedirs(OUT, exist_ok=True)
DATA = '/content/data'
print('OUT =', OUT)

In [ ]:
import os
if 'OUT' not in globals():
    OUT, DATA = '/content/bench_out', '/content/data'
    os.makedirs(OUT, exist_ok=True)
REPO = 'https://github.com/incognito-tab/XLDL.git'
REPO_DIR = '/content/XLDL'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1
!pip -q install spikingjelly==0.0.0.0.14 einops ml_collections timm h5py

In [ ]:
import shlex, subprocess, time

def run(args, time_limit=0, out=None):
    """Chạy một thí nghiệm; in log theo từng epoch."""
    cmd = f'pytho
    n -m bench.run {args} --out {out or OUT} --data-root {DATA}'
    if time_limit:
        cmd += f' --time-limit {time_limit}'
    print('\n$', cmd, flush=True)
    t = time.time()
    p = subprocess.Popen(shlex.split(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'warn' not in line.lower():
            print(line, end='', flush=True)
    code = p.wait()
    print(f'[exit {code}] {(time.time() - t) / 60:.1f} min', flush=True)
    return code

MAIN = [  # (dataset, model args)
    ('fmnist',        '--model mixer --mixer-size small --T 4'),
    ('fmnist',        '--model mixer --mixer-size lite --T 4'),
    ('fmnist',        '--model dmp'),
    ('fmnist_stream', '--model mixer --mixer-size lite'),
    ('fmnist_stream', '--model dmp'),
    ('cifar10',       '--model mixer --mixer-size small --T 4'),
    ('cifar10',       '--model mixer --mixer-size lite --T 4'),
    ('cifar10',       '--model dmp'),
    ('shd',           '--model mixer --mixer-size lite'),
    ('shd',           '--model dmp'),
]
DELAYS = [10, 25, 50, 100, 200]

## 0. Smoke test (~5–10 phút)
Mỗi cấu hình chạy 1 epoch trên 512 mẫu để chắc mọi thứ chạy được trên GPU, đồng thời tải dữ liệu (CIFAR-10 ~170MB, SHD ~170MB). **Xem cột `s/epoch`** để ước lượng thời gian thật (nhân với số mẫu thật / 512).

In [ ]:
SMOKE = '/content/smoke'
for ds, m in MAIN:
    run(f'--dataset {ds} {m} --epochs 1 --max-train 512', out=SMOKE)
run('--dataset recall --delay 50 --model mixer --mixer-size lite --epochs 1 --max-train 512', out=SMOKE)
run('--dataset recall --delay 50 --model dmp --epochs 1 --max-train 512', out=SMOKE)

## A. Bảng chính — seed 0

In [ ]:
for ds, m in MAIN:
    run(f'--dataset {ds} {m} --seed 0')

## B. Delayed recall — nhớ quá khứ theo độ trễ D
* `dmp` mặc định: cửa sổ nhớ `theta` = độ dài chuỗi (luôn đủ dài).
* `dmp --theta 50 --tag fixedtheta`: cửa sổ nhớ cố định 50 bước → kiểm tra điều gì xảy ra khi D > theta.

In [ ]:
for seed in (0, 1):
    for D in DELAYS:
        run(f'--dataset recall --delay {D} --model dmp --seed {seed}')
        run(f'--dataset recall --delay {D} --model dmp --theta 50 --tag fixedtheta --seed {seed}')
        run(f'--dataset recall --delay {D} --model mixer --mixer-size lite --seed {seed}')

## C. Ablation (seed 0)
* DMP **không có bộ nhớ chậm** (`--gamma 0`, train lại từ đầu) → bộ nhớ đóng góp bao nhiêu.
* DMP: độ nhạy với `d` (số chiều bộ nhớ) và `theta` (cửa sổ nhớ) trên SHD.
* Mixer: tăng hằng số thời gian LIF `tau` 2 → 8 (nhớ lâu hơn có giúp Mixer không?) và số bước `T` trên ảnh tĩnh.

In [ ]:
ABL = [
    '--dataset shd --model dmp --gamma 0',
    '--dataset fmnist_stream --model dmp --gamma 0',
    '--dataset recall --delay 50 --model dmp --gamma 0',
    *[f'--dataset shd --model dmp --d {d}' for d in (1, 2, 5, 20)],
    *[f'--dataset shd --model dmp --theta {t}' for t in (10, 20, 80, 100)],
    '--dataset shd --model mixer --mixer-size lite --tau 8',
    '--dataset fmnist_stream --model mixer --mixer-size lite --tau 8',
    '--dataset recall --delay 50 --model mixer --mixer-size lite --tau 8',
    *[f'--dataset cifar10 --model mixer --mixer-size lite --T {T}' for T in (1, 2, 8)],
]
for a in ABL:
    run(a + ' --seed 0')

## D. Thêm seed 1, 2 cho bảng chính (để có mean ± std)

In [ ]:
for seed in (1, 2):
    for ds, m in MAIN:
        run(f'--dataset {ds} {m} --seed {seed}')

## Tổng hợp

In [ ]:
!cd {REPO_DIR} && python -m bench.summarize --out {OUT}
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{OUT}/plots/*.png')):
    print(p); display(Image(p))

In [ ]:
# In toàn bộ kết quả dạng gọn (để Claude đọc trực tiếp từ notebook)
import json, glob
for p in sorted(glob.glob(f'{OUT}/results/*.json')):
    r = json.load(open(p))
    h = r.pop('history'); r.pop('args', None)
    r['val_curve'] = [round(e['val_acc'] * 100, 1) for e in h]
    r['train_curve'] = [round(e['train_acc'] * 100, 1) for e in h]
    print('RESULTJSON', json.dumps(r))